Чекпоинт 6

In [2]:
!pip install catboost

In [3]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import LeaveOneGroupOut
from scipy.spatial import cKDTree

In [4]:
df = pd.read_csv('/content/grid_features.csv')
df.head()

,lat,lon,city,atm_count,sber_count,tinkoff_count,vtb_count,alfa_count,gazprom_count,raiff_count,...,poi_diversity_500m,poi_diversity_1000m,total_poi_500m,total_poi_1000m,retail_share_500m,residential_ratio_500m,orgs_500m,org_diversity_500m,orgs_1000m,org_diversity_1000m
0,54.965929,36.668509,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,54.965929,36.676464,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,54.965929,36.684419,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,54.965929,36.692373,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,54.965929,36.700328,Москва,0,0,0,0,0,0,0,...,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Базовый прогон (позаимствован у Димы)

In [5]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)

# Убираем банкоматные счётчики из фичей (утечка) + lat/lon + target
drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols and c != 'city']

X = df[feature_cols].copy()
y = df['target'].values
groups = df['city'].values

# NaN в avg_levels - заполняем 0 (нет жилых домов)
X = X.fillna(0)

logo = LeaveOneGroupOut()
results = []

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    city = groups[test_idx[0]]
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    model = CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05,
        auto_class_weights='Balanced',
        eval_metric='AUC', verbose=100,
        random_seed=42
    )
    model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

    proba = model.predict_proba(X_test)[:, 1]
    roc = roc_auc_score(y_test, proba)
    pr = average_precision_score(y_test, proba)
    baseline = y_test.mean()

    results.append({'city': city, 'ROC-AUC': roc, 'PR-AUC': pr,
                    'baseline': baseline, 'PR/baseline': pr / baseline if baseline > 0 else 0})
    print(f"{city}: ROC-AUC={roc:.4f}, PR-AUC={pr:.4f}, PR/baseline={pr/baseline:.1f}x")

res = pd.DataFrame(results)
print("\nСРЕДНИЕ")
print(f"ROC-AUC:     {res['ROC-AUC'].mean():.4f}")
print(f"PR-AUC:      {res['PR-AUC'].mean():.4f}")
print(f"PR/baseline: {res['PR/baseline'].mean():.1f}x")

# Feature importance top-20
print("\nFEATURE IMPORTANCE (last fold)")
fi = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(fi.head(20).to_string())

0:	test: 0.9719680	best: 0.9719680 (0)	total: 242ms	remaining: 2m
100:	test: 0.9886336	best: 0.9886336 (100)	total: 23.4s	remaining: 1m 32s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9887925335
bestIteration = 130

Shrink model to first 131 iterations.
Казань: ROC-AUC=0.9888, PR-AUC=0.7628, PR/baseline=32.2x
0:	test: 0.9541021	best: 0.9541021 (0)	total: 218ms	remaining: 1m 48s
100:	test: 0.9845831	best: 0.9846080 (92)	total: 21s	remaining: 1m 23s
200:	test: 0.9852837	best: 0.9853176 (198)	total: 40.8s	remaining: 1m
Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.9854969164
bestIteration = 230

Shrink model to first 231 iterations.
Москва: ROC-AUC=0.9855, PR-AUC=0.7619, PR/baseline=19.6x
0:	test: 0.9842520	best: 0.9842520 (0)	total: 196ms	remaining: 1m 37s
100:	test: 0.9957733	best: 0.9957733 (100)	total: 20.5s	remaining: 1m 20s
200:	test: 0.9957828	best: 0.9958828 (166)	total: 41.9s	remaining: 1m 2s
Stopped by overfitting detector  (50 itera

In [6]:
!pip install h3 scikit-learn gensim node2vec -q
import pandas as pd
import numpy as np
import h3
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score
from catboost import CatBoostClassifier
import networkx as nx
from node2vec import Node2Vec
import warnings
warnings.filterwarnings('ignore')

In [7]:
df = pd.read_csv('grid_features.csv')
df['target'] = (df['atm_count'] > 0).astype(int)

drop_cols = ['atm_count', 'sber_count', 'tinkoff_count', 'vtb_count',
             'alfa_count', 'gazprom_count', 'raiff_count',
             'lat', 'lon', 'target']
feature_cols = [c for c in df.columns if c not in drop_cols and c != 'city']

X = df[feature_cols].fillna(0)
y = df['target'].values
groups = df['city'].values

Построю гексоганальную сетку

In [8]:
# Присваиваем каждой точке индекс гексагона (H3 resolution=8)
df['hex_id'] = df.apply(
    lambda row: h3.latlng_to_cell(row['lat'], row['lon'], res=8), axis=1
)

# Создадим набор уникальных гексагонов и для каждого агрегируем средние значения признаков
hex_groups = df.groupby('hex_id')
hex_feature_means = hex_groups[feature_cols].mean()
hex_feature_means.head()

,metro_500m,metro_1000m,nearest_metro,bus_stops_500m,bus_stops_1000m,nearest_bus_stops,malls_500m,malls_1000m,nearest_malls,business_centres_500m,...,poi_diversity_500m,poi_diversity_1000m,total_poi_500m,total_poi_1000m,retail_share_500m,residential_ratio_500m,orgs_500m,org_diversity_500m,orgs_1000m,org_diversity_1000m
hex_id,,,,,,,,,,,,,,,,,,,,,
880b86d001fffff,0.0,0.0,112040.355024,0.0,0.333333,1146.515925,0.0,0.0,7289.633704,0.0,...,0.0,0.333333,0.0,0.333333,0.0,0.0,0.0,0.0,0.333333,0.333333
880b86d003fffff,0.0,0.0,113025.418017,0.0,1.000000,686.199942,0.0,0.0,6697.828604,0.0,...,0.0,1.500000,0.0,1.500000,0.0,0.0,1.0,1.0,1.500000,1.500000
880b86d005fffff,0.0,0.0,111747.469624,0.0,0.000000,1954.817773,0.0,0.0,8138.587256,0.0,...,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0,0.0,0.000000,0.000000
880b86d007fffff,0.0,0.0,112852.056149,0.0,0.666667,937.971963,0.0,0.0,7697.357631,0.0,...,0.0,1.333333,0.0,1.333333,0.0,0.0,0.0,0.0,1.333333,1.333333
880b86d009fffff,0.0,0.0,111291.920029,0.0,0.500000,1173.887103,0.0,0.0,6822.629706,0.0,...,0.0,0.500000,0.0,0.500000,0.0,0.0,0.0,0.0,0.000000,0.000000


In [9]:
G = nx.Graph()
for h in hex_feature_means.index:
    G.add_node(h)

for h in G.nodes:
    # grid_disk возвращает итератор, преобразуем в множество
    neighbors = set(h3.grid_disk(h, 1)) - {h}
    for n in neighbors:
        if n in G:
            G.add_edge(h, n)

print(f"Граф: {len(G.nodes)} узлов, {len(G.edges)} рёбер")

Граф: 173114 узлов, 515712 рёбер


In [ ]:
import numpy as np
from gensim.models import Word2Vec

# Параметры
walk_length = 20
num_walks = 40
dimensions = 32
seed = 42
np.random.seed(seed)

# Генерация случайных блужданий
walks = []
for node in G.nodes:
    for _ in range(num_walks):
        walk = [str(node)]  # gensim ожидает строки
        current = node
        for _ in range(walk_length - 1):
            nbrs = list(G.neighbors(current))
            if not nbrs:
                break
            current = np.random.choice(nbrs)
            walk.append(str(current))
        walks.append(walk)

# Обучение Word2Vec (Skip-Gram)
w2v_model = Word2Vec(
    sentences=walks,
    vector_size=dimensions,
    window=10,
    sg=0,               # Skip-Gram
    hs=0,               # negative sampling
    negative=5,
    min_count=1,
    workers=4,          # можно увеличить, если не возникает проблем
    epochs=3,
    seed=seed
)

# Извлекаем эмбеддинги
hex_embeddings = {node: w2v_model.wv[str(node)] for node in G.nodes()}
emb_dim = dimensions

print(f"Размерность эмбеддинга: {emb_dim}, пример для первого гексагона: {list(hex_embeddings.values())[0][:5]} ...")

у меня не хватило памяти по часу 3 раза ждала и каждый раз падало с ошибкой(((